# Predicting the Electronic Band Gap of Inorganic Crystalline Materials from Composition and Structure
### Stage 5 — Feature Engineering

Uses `mp_bandgap_stage4_cleaned.csv` (153,597 rows) from Stage 4. This notebook builds:

1. Composition-derived descriptors (elemental statistics computed from each material's formula)
2. Structure-derived descriptors (from the summary fields already in the cleaned dataset)
3. Two parallel feature tables — composition-only and composition+structure — so Stage 6/7 can directly test whether structural information adds predictive value (Stage 2, secondary RQ1)

**No train/test split, no model fitting, no performance numbers here.** This ends with saved feature tables and a feature dictionary explaining the chemical meaning of every descriptor.


## Load the Stage 4 output

Same Drive-fallback pattern as Stage 4: tries local storage first, then `MyDrive/materials_bandgap_project/` on Google Drive.


In [ ]:
!pip install -q pymatgen pandas numpy scikit-learn


In [ ]:
import os
import json
import ast
import shutil
import time

import numpy as np
import pandas as pd
from pymatgen.core import Composition, Element

DRIVE_DIR = "/content/drive/MyDrive/materials_bandgap_project"
CLEANED_CSV_NAME = "mp_bandgap_stage4_cleaned.csv"
CLEANED_CSV_PATH = CLEANED_CSV_NAME

if not os.path.exists(CLEANED_CSV_PATH):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        pass

    drive_path = os.path.join(DRIVE_DIR, CLEANED_CSV_NAME)
    if os.path.exists(drive_path):
        shutil.copy(drive_path, CLEANED_CSV_PATH)
        print(f"Loaded {CLEANED_CSV_NAME} from Google Drive.")
    else:
        raise FileNotFoundError(
            f"{CLEANED_CSV_NAME} not found locally or at {drive_path} on Drive. "
            "Run Stage 4 first, and/or save its output to that Drive folder."
        )

df = pd.read_csv(CLEANED_CSV_PATH)

def parse_elements(val):
    if pd.isna(val):
        return None
    try:
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        return None

df["elements"] = df["elements"].apply(parse_elements)

# pandas reads the literal text "NaN" (the compound sodium nitride, formula NaN) as a
# missing value during CSV round-trips. Restore it, but only after verifying that every
# affected row really is a Na-N compound -- otherwise stop rather than guess.
missing_formula_mask = df["formula_pretty"].isna()
n_missing_formula = int(missing_formula_mask.sum())
print(f"Rows with missing formula_pretty after CSV load: {n_missing_formula}")
if n_missing_formula:
    sub = df[missing_formula_mask]
    all_na_n = all(set(els) == {"Na", "N"} for els in sub["elements"] if isinstance(els, list)) \
        and sub["elements"].apply(lambda e: isinstance(e, list)).all() \
        and (sub["nelements"] == 2).all()
    assert all_na_n, "Missing formulas are NOT all Na-N compounds -- investigate before proceeding."
    df.loc[missing_formula_mask, "formula_pretty"] = "NaN"
    print(f"Restored formula 'NaN' (sodium nitride) for {n_missing_formula} rows; verified all are 2-element Na/N entries.")

print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns.")
df.head()


## Step 5.1 — Build an Elemental Property Table

For every unique element present in the dataset, we pull four properties from pymatgen's `Element` class: atomic number (`Z`), atomic mass (`atomic_mass`), Pauling electronegativity (`X`), and atomic radius (`atomic_radius`). These are looked up once per element (not per row) and cached in a small table.

`atomic_radius` is known to be undefined (`None`) for some elements in pymatgen (particularly some noble gases and less common elements) — we check for this explicitly rather than assuming it's always populated, and report exactly which elements are affected before deciding how to handle it.


In [ ]:
all_elements = sorted(set().union(*df["elements"].dropna()))
print(f"Unique elements in dataset: {len(all_elements)}")

element_rows = []
for el_symbol in all_elements:
    el = Element(el_symbol)
    element_rows.append({
        "symbol": el_symbol,
        "Z": el.Z,
        "atomic_mass": float(el.atomic_mass),
        "electronegativity": float(el.X) if el.X is not None else np.nan,
        "atomic_radius": float(el.atomic_radius) if el.atomic_radius is not None else np.nan,
    })

element_props = pd.DataFrame(element_rows).set_index("symbol")

missing_radius = element_props[element_props["atomic_radius"].isna()].index.tolist()
missing_electroneg = element_props[element_props["electronegativity"].isna()].index.tolist()

print(f"\nElements missing atomic_radius ({len(missing_radius)}): {missing_radius}")
print(f"Elements missing electronegativity ({len(missing_electroneg)}): {missing_electroneg}")

element_props.to_csv("element_property_table.csv")
element_props.head()


**Handling missing elemental properties:** rather than dropping every material containing an element with an undefined property (which could silently remove specific chemistries, biasing the dataset), any compound-level descriptor built from a property with a missing value is left as `NaN` for that specific descriptor, for that specific material — not imputed with a guessed number, and not used as a reason to drop the row entirely. `HistGradientBoostingRegressor` and tree-based models can handle `NaN` natively; this is revisited explicitly if it turns out to affect a large fraction of the dataset (checked below).


## Step 5.2 — Composition-Derived Descriptors

For each material, its formula (`formula_pretty`) is parsed into a fractional composition (element → fraction of atoms). From that, for each of the four elemental properties above, we compute:

- **mean** (fraction-weighted average across constituent elements) — the "typical" value for this compound
- **std** (fraction-weighted standard deviation) — how much the constituent elements vary in this property
- **min / max / range** (unweighted extremes across the *distinct* elements present) — captures the most extreme contrast in the compound, regardless of stoichiometric weight

Plus three composition-level descriptors that aren't tied to a single elemental property:

- **`n_elements`** — already available, included here for completeness
- **`frac_transition_metal`** — the fraction of atoms that are d-block (transition metal) elements. Chemical meaning: proxy for d-orbital involvement in bonding/electronic structure, directly relevant to the coordination-chemistry interpretation planned for Stage 9.
- **`frac_main_group`** — fraction of atoms that are s-block or p-block elements (the complement of transition metals and f-block/lanthanide-actinide elements).
- **`composition_entropy`** — Shannon entropy of the elemental fractions, $-\sum_i f_i \ln f_i$. Chemical meaning: a diversity/disorder measure — low for simple binary compounds, higher for compounds with many elements in more even proportions.

**Important limitation to note explicitly:** composition-only descriptors are, by construction, identical for every polymorph of the same formula. This is expected, not a bug — it's exactly what lets Stage 6/7 test whether adding structural descriptors improves predictions beyond what composition alone can explain.


In [ ]:
# Same transition-metal / main-group classification used consistently since Stage 3/4
D_BLOCK = set([
    "Sc","Ti","V","Cr","Mn","Fe","Co","Ni","Cu","Zn",
    "Y","Zr","Nb","Mo","Tc","Ru","Rh","Pd","Ag","Cd",
    "Hf","Ta","W","Re","Os","Ir","Pt","Au","Hg",
])
F_BLOCK = set([
    "La","Ce","Pr","Nd","Pm","Sm","Eu","Gd","Tb","Dy","Ho","Er","Tm","Yb","Lu",
    "Ac","Th","Pa","U","Np","Pu","Am","Cm","Bk","Cf","Es","Fm","Md","No","Lr",
])

PROPERTIES = ["Z", "atomic_mass", "electronegativity", "atomic_radius"]

def composition_features(formula: str):
    """Returns a dict of composition-derived descriptors for one formula string.
    Returns None if the formula fails to parse (logged and counted by the caller)."""
    comp = Composition(formula)
    frac = comp.fractional_composition.get_el_amt_dict()  # {symbol: fraction}, sums to 1

    feats = {"n_elements": len(frac)}

    frac_tm = sum(f for el, f in frac.items() if el in D_BLOCK)
    frac_main_group = sum(f for el, f in frac.items() if el not in D_BLOCK and el not in F_BLOCK)
    feats["frac_transition_metal"] = frac_tm
    feats["frac_main_group"] = frac_main_group

    fracs = np.array(list(frac.values()))
    feats["composition_entropy"] = float(-(fracs * np.log(fracs)).sum())

    for prop in PROPERTIES:
        values = np.array([element_props.loc[el, prop] for el in frac.keys()], dtype=float)
        weights = np.array([frac[el] for el in frac.keys()], dtype=float)

        if np.isnan(values).all():
            feats[f"{prop}_mean"] = np.nan
            feats[f"{prop}_std"] = np.nan
            feats[f"{prop}_min"] = np.nan
            feats[f"{prop}_max"] = np.nan
            feats[f"{prop}_range"] = np.nan
            continue

        valid = ~np.isnan(values)
        v, w = values[valid], weights[valid]
        w = w / w.sum()  # renormalize weights over the elements with a valid property value

        w_mean = float((v * w).sum())
        w_std = float(np.sqrt((w * (v - w_mean) ** 2).sum()))

        feats[f"{prop}_mean"] = w_mean
        feats[f"{prop}_std"] = w_std
        feats[f"{prop}_min"] = float(v.min())
        feats[f"{prop}_max"] = float(v.max())
        feats[f"{prop}_range"] = float(v.max() - v.min())

    return feats


## Step 5.3 — Apply Across the Dataset (cached by unique formula)

Since composition descriptors depend only on the formula, not the specific structure, we compute them once per unique formula and map the result onto every row — this is both a large speed-up (there are far fewer unique formulas than rows, per the Stage 4 duplicate check) and a direct consequence of the point above: identical formulas necessarily get identical composition features.


In [ ]:
unique_formulas = df["formula_pretty"].unique()
print(f"Computing composition descriptors for {len(unique_formulas)} unique formulas "
      f"(covering {df.shape[0]} rows)...")

formula_feature_cache = {}
failed_formulas = []

start = time.time()
for formula in unique_formulas:
    try:
        formula_feature_cache[formula] = composition_features(formula)
    except Exception as e:
        failed_formulas.append((formula, str(e)))

elapsed = time.time() - start
print(f"Done in {elapsed:.1f}s. Failed to parse: {len(failed_formulas)} formulas.")
if failed_formulas:
    print("Examples of failures (up to 10):", failed_formulas[:10])


In [ ]:
# Build the feature table from the cache directly (robust to failed formulas,
# which simply get an all-NaN row instead of breaking the column structure)
comp_feature_table = pd.DataFrame.from_dict(formula_feature_cache, orient="index")
comp_feature_names = list(comp_feature_table.columns)

comp_feature_df = comp_feature_table.reindex(df["formula_pretty"].values)
comp_feature_df.index = df.index

n_rows_with_failed_formula = df["formula_pretty"].isin([f for f, _ in failed_formulas]).sum()
print(f"Formulas that failed to parse: {len(failed_formulas)}")
print(f"Rows affected by a formula-parse failure: {n_rows_with_failed_formula} "
      f"(their composition features are NaN, rows are NOT dropped)")

missing_by_feature = comp_feature_df.isna().sum()
missing_by_feature = missing_by_feature[missing_by_feature > 0]
print("\nComposition features with missing values:")
print(missing_by_feature if len(missing_by_feature) else "(none)")

comp_feature_df.describe().T


## Step 5.4 — Structure-Derived Descriptors

These come directly from the Stage 3/4 summary fields — no new computation needed, just a clear encoding decision:

- `density`, `density_atomic`, `volume`, `nsites` — used as-is (already numeric and physically meaningful)
- `crystal_system` — one-hot encoded (7 categories: cubic, hexagonal, monoclinic, orthorhombic, tetragonal, triclinic, trigonal). This is a genuine categorical variable — there's no meaningful numeric ordering between crystal systems.
- `space_group_number` — **deliberately excluded from the model-ready feature set**, even though it's numeric. Space group numbers (1–230) are an arbitrary cataloguing convention (International Tables for Crystallography), not an ordinal physical scale — treating "space group 200" as "twice" space group 100 would be numerically meaningless and could mislead a model. It's kept in the saved dataset for reference/lookup, just not passed to the models.


In [ ]:
crystal_system_dummies = pd.get_dummies(df["crystal_system"], prefix="crystal_system")
print(f"Crystal system categories found: {list(df['crystal_system'].dropna().unique())}")
print(f"Missing crystal_system: {df['crystal_system'].isna().sum()} rows")

structure_feature_names = ["density", "density_atomic", "volume", "nsites"] + list(crystal_system_dummies.columns)
structure_feature_df = pd.concat(
    [df[["density", "density_atomic", "volume", "nsites"]], crystal_system_dummies],
    axis=1,
)
structure_feature_df.describe().T


## Step 5.5 — Assemble Final Feature Tables

Two parallel tables, sharing the same row index and identifier/target columns, so Stage 6/7 can train the same model type on each and directly compare:


In [ ]:
id_target_cols = ["material_id", "formula_pretty", "band_gap", "energy_above_hull", "is_stable"]

composition_only_df = pd.concat([df[id_target_cols], comp_feature_df], axis=1)
composition_structure_df = pd.concat([df[id_target_cols], comp_feature_df, structure_feature_df], axis=1)

print("Composition-only feature table:", composition_only_df.shape)
print("Composition+structure feature table:", composition_structure_df.shape)

composition_only_df.to_csv("mp_bandgap_features_composition_only.csv", index=False)
composition_structure_df.to_csv("mp_bandgap_features_composition_structure.csv", index=False)

print("\nSaved both feature tables.")


## Step 5.6 — Feature Dictionary (chemical meaning of every descriptor)

Written alongside the feature tables so the eventual README/report doesn't need to reconstruct this from code later.


In [ ]:
feature_dictionary = {
    "n_elements": "Number of distinct elements in the formula. Proxy for compositional complexity.",
    "frac_transition_metal": "Fraction of atoms that are d-block (transition metal) elements. Proxy for d-orbital involvement in bonding.",
    "frac_main_group": "Fraction of atoms that are s-block or p-block (main-group) elements.",
    "composition_entropy": "Shannon entropy of elemental fractions (-sum f_i ln f_i). Compositional diversity/disorder measure.",
    "Z_mean": "Composition-weighted mean atomic number. Rough proxy for average electron count/size of constituent atoms.",
    "Z_std": "Composition-weighted spread in atomic number across constituent elements.",
    "Z_min": "Smallest atomic number among constituent elements (unweighted).",
    "Z_max": "Largest atomic number among constituent elements (unweighted).",
    "Z_range": "Range (max-min) of atomic number among constituent elements.",
    "atomic_mass_mean": "Composition-weighted mean atomic mass (amu).",
    "atomic_mass_std": "Composition-weighted spread in atomic mass.",
    "atomic_mass_min": "Smallest atomic mass among constituent elements.",
    "atomic_mass_max": "Largest atomic mass among constituent elements.",
    "atomic_mass_range": "Range of atomic mass among constituent elements.",
    "electronegativity_mean": "Composition-weighted mean Pauling electronegativity. Relates to bond ionicity/polarity, chemically linked to band gap size.",
    "electronegativity_std": "Composition-weighted spread in electronegativity across elements.",
    "electronegativity_min": "Smallest electronegativity among constituent elements.",
    "electronegativity_max": "Largest electronegativity among constituent elements.",
    "electronegativity_range": "Electronegativity contrast between the most and least electronegative constituent element -- a common proxy for bond ionicity.",
    "atomic_radius_mean": "Composition-weighted mean atomic radius (Angstrom).",
    "atomic_radius_std": "Composition-weighted spread in atomic radius.",
    "atomic_radius_min": "Smallest atomic radius among constituent elements.",
    "atomic_radius_max": "Largest atomic radius among constituent elements.",
    "atomic_radius_range": "Range of atomic radius among constituent elements. Related to size mismatch/strain in the structure.",
    "density": "Bulk density of the relaxed structure (g/cm3). Structure-derived.",
    "density_atomic": "Atomic density (atoms per unit volume). Structure-derived.",
    "volume": "Unit cell volume (Angstrom^3). Structure-derived.",
    "nsites": "Number of atomic sites in the unit cell. Structure-derived.",
    "crystal_system_*": "One-hot encoded crystal system (cubic, hexagonal, monoclinic, orthorhombic, tetragonal, triclinic, trigonal). Structure-derived, categorical.",
    "energy_above_hull": "Thermodynamic metastability (eV/atom). Retained as an available covariate, NOT used by default per the Stage 2 leakage review (test cautiously, not causal).",
    "is_stable": "Whether the material sits exactly on the convex hull. Retained for reference, same caution as energy_above_hull.",
}

with open("feature_dictionary.json", "w") as f:
    json.dump(feature_dictionary, f, indent=2)

print(f"Feature dictionary written with {len(feature_dictionary)} entries.")
print(json.dumps(feature_dictionary, indent=2))


## Step 5.7 — Leakage Re-Check and Sanity Correlation Table

Before finishing, a quick re-confirmation that nothing in the engineered feature set traces back to `band_gap`, `cbm`, `vbm`, `efermi`, `is_metal`, or `is_gap_direct` (all excluded per the Stage 2 leakage review) — and a plain correlation table against the target, purely as a sanity check for anything absurdly high that would suggest an error, not as a feature-importance result.


In [ ]:
assert not any(c in composition_structure_df.columns for c in ["cbm", "vbm", "efermi", "is_metal", "is_gap_direct"]), \
    "A field flagged as leakage risk in Stage 2 ended up in the feature table -- stop and investigate."

numeric_feature_cols = [c for c in composition_structure_df.columns
                         if c not in ["material_id", "formula_pretty", "band_gap"]]

correlations = composition_structure_df[numeric_feature_cols + ["band_gap"]].corr()["band_gap"].drop("band_gap")
correlations = correlations.sort_values(key=abs, ascending=False)

print("Pearson correlation of each engineered feature with band_gap (sanity check only, not a modeling result):")
print(correlations)


In [ ]:
# Persist Stage 5 outputs to Google Drive so Stage 6 can find them
import shutil, os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = "/content/drive/MyDrive/materials_bandgap_project"
    os.makedirs(SAVE_DIR, exist_ok=True)
    for fname in ["mp_bandgap_features_composition_only.csv",
                  "mp_bandgap_features_composition_structure.csv",
                  "feature_dictionary.json",
                  "element_property_table.csv"]:
        if os.path.exists(fname):
            shutil.copy(fname, os.path.join(SAVE_DIR, fname))
            print(f"Copied {fname} to Drive")
        else:
            print(f"WARNING: {fname} not found in this session")
except ImportError:
    print("Not running in Colab -- files remain in the local working directory.")


### Stage 5 ends here.

No train/test split, no baseline model, no ML model has been built. Files produced:
- `mp_bandgap_features_composition_only.csv`
- `mp_bandgap_features_composition_structure.csv`
- `feature_dictionary.json`
- `element_property_table.csv`

**Do not run Stage 6 code in this notebook.** Await confirmation before proceeding — Stage 6 is where the grouped train/test split and baseline models are actually built.
